In [1]:
import pandas as pd
import glob
import os
import sys

# Define directories
data_dir = r"c:\Users\Ashwa\Ash_projects\Indigo Research\BACIDataset1995"
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

# Step 1: Load and Aggregate BACI Data
# File pattern for the user's dataset
baci_files = sorted(glob.glob(os.path.join(data_dir, "BACI_HS92_Y*_V202601.csv")))
print(f"Found {len(baci_files)} BACI files.")

if not baci_files:
    print("No BACI files found! Please check the dataset path.")
    sys.exit(1)

# Only process years 1995-2022 as per plan logic?
# The plan mentions 1995–2022. Let's filter if needed, 
# although having more years (up to 2024) might be better. 
# Let's use all available files to be comprehensive.

df_list = []
for file in baci_files:
    print(f"Loading {os.path.basename(file)}...")
    df = pd.read_csv(file)
    # Aggregate: X_cpt = sum over j (all importers)
    # Columns in BACI are t, i, j, k, v, q
    # We want to group by year (t), exporter (i), product (k)
    exports_year = df.groupby(['t', 'i', 'k'])['v'].sum().reset_index()
    exports_year.columns = ['year', 'country', 'product', 'value']
    df_list.append(exports_year)

# Concatenate all years
print("Concatenating aggregated data...")
exports = pd.concat(df_list, ignore_index=True)

# Save intermediate result
output_file = os.path.join(output_dir, "exports_cpt.csv")
exports.to_csv(output_file, index=False)

# Validation Statistics
print("\nValidation Statistics for Step 1:")
print(f"Output file: {output_file}")
print(f"File created: {os.path.exists(output_file)}")
print(f"Total rows: {len(exports)}")
print(f"Year range: {exports['year'].min()} to {exports['year'].max()}")
print(f"Unique countries: {exports['country'].nunique()}")
print(f"Unique products: {exports['product'].nunique()}")
print(f"Total trade value: {exports['value'].sum():,.0f}")


Found 30 BACI files.
Loading BACI_HS92_Y1995_V202601.csv...
Loading BACI_HS92_Y1996_V202601.csv...
Loading BACI_HS92_Y1997_V202601.csv...
Loading BACI_HS92_Y1998_V202601.csv...
Loading BACI_HS92_Y1999_V202601.csv...
Loading BACI_HS92_Y2000_V202601.csv...
Loading BACI_HS92_Y2001_V202601.csv...
Loading BACI_HS92_Y2002_V202601.csv...
Loading BACI_HS92_Y2003_V202601.csv...
Loading BACI_HS92_Y2004_V202601.csv...
Loading BACI_HS92_Y2005_V202601.csv...
Loading BACI_HS92_Y2006_V202601.csv...
Loading BACI_HS92_Y2007_V202601.csv...
Loading BACI_HS92_Y2008_V202601.csv...
Loading BACI_HS92_Y2009_V202601.csv...
Loading BACI_HS92_Y2010_V202601.csv...
Loading BACI_HS92_Y2011_V202601.csv...
Loading BACI_HS92_Y2012_V202601.csv...
Loading BACI_HS92_Y2013_V202601.csv...
Loading BACI_HS92_Y2014_V202601.csv...
Loading BACI_HS92_Y2015_V202601.csv...
Loading BACI_HS92_Y2016_V202601.csv...
Loading BACI_HS92_Y2017_V202601.csv...
Loading BACI_HS92_Y2018_V202601.csv...
Loading BACI_HS92_Y2019_V202601.csv...
Load

In [2]:
import pandas as pd
import numpy as np
import os

# Define directories
output_dir = "data"
input_file = os.path.join(output_dir, "exports_cpt.csv")
output_file = os.path.join(output_dir, "rca_cpt.csv")

print(f"Starting Step 2: Computing RCA Matrix...")

if not os.path.exists(input_file):
    print(f"Error: {input_file} not found. Did you run Step 1?")
    exit(1)

# Load aggregated exports
exports = pd.read_csv(input_file)

# Compute totals per year for RCA formula
# Formula: (X_cpt / Sum_p X_cpt) / (Sum_c X_cpt / Sum_cp X_cpt)
exports['country_total'] = exports.groupby(['year', 'country'])['value'].transform('sum')
exports['product_total'] = exports.groupby(['year', 'product'])['value'].transform('sum')
exports['world_total'] = exports.groupby('year')['value'].transform('sum')

# Compute RCA
# Adding a small epsilon 1e-8 to avoid division by zero
exports['rca'] = (exports['value'] / exports['country_total']) / \
                 ((exports['product_total'] + 1e-8) / exports['world_total'])

# Drop intermediate columns
exports = exports[['year', 'country', 'product', 'value', 'rca']]

# Save RCA matrix
exports.to_csv(output_file, index=False)

# Validation Statistics
print("\nValidation Statistics for Step 2:")
print(f"Output file: {output_file}")
print(f"File created: {os.path.exists(output_file)}")
print(f"Total rows: {len(exports)}")
print(f"Average RCA: {exports['rca'].mean():.4f}")
print(f"Max RCA: {exports['rca'].max():.4f}")
print(f"RCA >= 1 entries: {(exports['rca'] >= 1).sum()}")


Starting Step 2: Computing RCA Matrix...

Validation Statistics for Step 2:
Output file: data\rca_cpt.csv
File created: True
Total rows: 14917868
Average RCA: 3.7590
Max RCA: 2443744.0005
RCA >= 1 entries: 2855403


In [1]:
import pandas as pd
import torch
import os

# Define directories
output_dir = "data"
input_file = os.path.join(output_dir, "rca_cpt.csv")
output_file = os.path.join(output_dir, "M_cpt_smoothed.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Starting Step 3 (CUDA Accelerated): Smoothing and Binarizing...")

if not os.path.exists(input_file):
    print(f"Error: {input_file} not found. Did you run Step 2?")
    exit(1)

# Load RCA data
df = pd.read_csv(input_file)

# 1. Map IDs to fixed indices for Tensor operations
years = sorted(df['year'].unique())
countries = sorted(df['country'].unique())
products = sorted(df['product'].unique())

year_map = {y: i for i, y in enumerate(years)}
country_map = {c: i for i, c in enumerate(countries)}
product_map = {p: i for i, p in enumerate(products)}

# 2. Create a sparse representation using a 3D Binary Tensor: [Years, Countries, Products]
print(f"Allocating 3D Tensor: [{len(years)}, {len(countries)}, {len(products)}] on {device}")
# M_binary: initialized to zero
M_binary = torch.zeros((len(years), len(countries), len(products)), dtype=torch.float32, device=device)

# Fill indices where RCA >= 1
mask = df['rca'] >= 1.0
active_df = df[mask]

# Extract indices for mapping
idx_y = torch.tensor(active_df['year'].map(year_map).values, device=device)
idx_c = torch.tensor(active_df['country'].map(country_map).values, device=device)
idx_p = torch.tensor(active_df['product'].map(product_map).values, device=device)

# Set RCA=1 markers in tensor
M_binary[idx_y, idx_c, idx_p] = 1.0

# 3. Apply 3-Year Rolling Window Sum (CUDA Accelerated)
print("Applying 3-year rolling window sum using 1D Convolution...")
# Reshape to [C*P, 1, Y] for 1D convolution across the Year axis
# (Treating each country-product pair as a 1D time series)
M_reshaped = M_binary.permute(1, 2, 0).reshape(-1, 1, len(years))

# Convolution kernel of [1, 1, 3] ones (sum of previous years)
# (Padding=1 to keep same length, though we should ignore first few years)
kernel = torch.ones((1, 1, 3), device=device)
M_sum = torch.nn.functional.conv1d(M_reshaped, kernel, padding=1)

# M=1 if sum >= 2 (RCA >= 1 in at least 2 out of 3 years)
M_smoothed = (M_sum >= 2.0).float().reshape(len(countries), len(products), len(years)).permute(2, 0, 1)

# 4. Filter back to sparse long-format and save
# Find indices where M_smoothed == 1
nonzero_indices = torch.nonzero(M_smoothed)
print(f"Found {len(nonzero_indices)} edges with RCA >= 1 (smoothed).")

# Convert back to DataFrame
edge_list = {
    'year': [years[i] for i in nonzero_indices[:, 0].cpu()],
    'country': [countries[i] for i in nonzero_indices[:, 1].cpu()],
    'product': [products[i] for i in nonzero_indices[:, 2].cpu()],
    'M': [1] * len(nonzero_indices)
}

edges_df = pd.DataFrame(edge_list)
edges_df.to_csv(output_file, index=False)

# Validation Statistics
print("\nValidation Statistics for Step 3:")
print(f"Output: {output_file}")
print(f"Total smoothed edges: {len(edges_df)}")
print("Smoothing (CUDA) Finished.")


Starting Step 3 (CUDA Accelerated): Smoothing and Binarizing...
Allocating 3D Tensor: [30, 233, 5018] on cuda
Applying 3-year rolling window sum using 1D Convolution...
Found 2549399 edges with RCA >= 1 (smoothed).

Validation Statistics for Step 3:
Output: data\M_cpt_smoothed.csv
Total smoothed edges: 2549399
Smoothing (CUDA) Finished.


In [2]:
import pandas as pd
import torch
import os
import random

# Define directories
output_dir = "data"
input_file = os.path.join(output_dir, "M_cpt_smoothed.csv")
output_file = os.path.join(output_dir, "labels_h5.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Starting Step 4 (CUDA Accelerated): Generating Prediction Labels (h=5)...")

if not os.path.exists(input_file):
    print(f"Error: {input_file} not found. Did you run Step 3?")
    exit(1)

# Load smoothed edges (only where M=1)
df = pd.read_csv(input_file)

# We'll re-convert to a 3D Tensor [Years, Countries, Products] on GPU
years = sorted(df['year'].unique())
countries = sorted(df['country'].unique())
products = sorted(df['product'].unique())

year_map = {y: i for i, y in enumerate(years)}
country_map = {c: i for i, c in enumerate(countries)}
product_map = {p: i for i, p in enumerate(products)}

print(f"Allocating 3D Tensor on {device}...")
M = torch.zeros((len(years), len(countries), len(products)), dtype=torch.uint8, device=device)

# Fill indices
idx_y = torch.tensor(df['year'].map(year_map).values, device=device)
idx_c = torch.tensor(df['country'].map(country_map).values, device=device)
idx_p = torch.tensor(df['product'].map(product_map).values, device=device)
M[idx_y, idx_c, idx_p] = 1

# Generate labels (h=5, horizon target t+5, sustained at t+6)
h = 5
total_years = len(years)
# We can only generate labels for years t where t+h+1 < total_years
limit = total_years - h - 1

print(f"Generating labels for {limit} years on CUDA...")

# 1. Slice Tensors for t, t+5, t+6
# Masks [T_valid, C, P]
M_t = M[:limit]
M_t5 = M[h:limit+h]
M_t6 = M[h+1:limit+h+1]

# 2. Extract Positive Indices (new sustained activation)
positives_mask = (M_t == 0) & (M_t5 == 1) & (M_t6 == 1)
pos_indices = torch.nonzero(positives_mask)
print(f"Found {len(pos_indices)} positive instances.")

# 3. Extract Negative Indices (stay 0)
negatives_mask = (M_t == 0) & (M_t5 == 0)
# Instead of keeping all (millions), we'll sample negative indices
# For efficiency, we'll sub-sample directly on GPU if possible
# Or just get all and sample in pandas
neg_indices = torch.nonzero(negatives_mask)
print(f"Total possible negative pool: {len(neg_indices)} instances.")

# --- Sampling ---
# We want roughly 2x negatives relative to positives
target_neg = min(len(neg_indices), 2 * len(pos_indices))
print(f"Sampling {target_neg} negatives to balance classes...")

# GPU sampling indices
sample_idx = torch.randperm(len(neg_indices), device=device)[:target_neg]
neg_sampled = neg_indices[sample_idx]

# --- Combine and Convert back ---
print("Converting back to long format and saving...")
all_labels = []

# Positives
pos_df = pd.DataFrame({
    'year': [years[i] for i in pos_indices[:, 0].cpu()],
    'country': [countries[i] for i in pos_indices[:, 1].cpu()],
    'product': [products[i] for i in pos_indices[:, 2].cpu()],
    'label': 1
})

# Negatives
neg_df = pd.DataFrame({
    'year': [years[i] for i in neg_sampled[:, 0].cpu()],
    'country': [countries[i] for i in neg_sampled[:, 1].cpu()],
    'product': [products[i] for i in neg_sampled[:, 2].cpu()],
    'label': 0
})

final_df = pd.concat([pos_df, neg_df], ignore_index=True)
final_df.to_csv(output_file, index=False)

# Validation Statistics
print("\nValidation Statistics for Step 4:")
print(f"Output: {output_file}")
print(f"Positive samples: {len(pos_df)}")
print(f"Negative samples: {len(neg_df)}")
print("Label generation (CUDA) Finished.")


Starting Step 4 (CUDA Accelerated): Generating Prediction Labels (h=5)...
Allocating 3D Tensor on cuda...
Generating labels for 24 years on CUDA...
Found 516327 positive instances.
Total possible negative pool: 25360403 instances.
Sampling 1032654 negatives to balance classes...
Converting back to long format and saving...

Validation Statistics for Step 4:
Output: data\labels_h5.csv
Positive samples: 516327
Negative samples: 1032654
Label generation (CUDA) Finished.


In [3]:
import pandas as pd
import numpy as np
import os

# Define directories
output_dir = "data"
input_file = os.path.join(output_dir, "rca_cpt.csv")
country_output = os.path.join(output_dir, "country_features.csv")
product_output = os.path.join(output_dir, "product_features.csv")

print(f"Starting Step 5: Creating Node Features...")

if not os.path.exists(input_file):
    print(f"Error: {input_file} not found. Did you run Step 2?")
    exit(1)

# Load RCA aggregated exports
exports = pd.read_csv(input_file)

# Country Features per year
print("Creating country features per year...")
country_features = []
countries = sorted(exports['country'].unique())
years = sorted(exports['year'].unique())

for year in years:
    year_data = exports[exports['year'] == year]
    # Total export value (log scale) per country
    c_totals = year_data.groupby('country')['value'].sum()
    # Number of products with RCA >= 1
    c_rca_count = year_data[year_data['rca'] >= 1].groupby('country').size()
    # Average RCA across all products
    c_avg_rca = year_data.groupby('country')['rca'].mean()
    # Max RCA
    c_max_rca = year_data.groupby('country')['rca'].max()

    for c in countries:
        features = {
            'year': year,
            'country': c,
            'log_export': np.log1p(c_totals.get(c, 0)),
            'n_products': c_rca_count.get(c, 0),
            'avg_rca': c_avg_rca.get(c, 0),
            'max_rca': c_max_rca.get(c, 0),
        }
        country_features.append(features)

country_feat_df = pd.DataFrame(country_features)

# Normalize country features (z-score per year)
print("Normalizing country features...")
for col in ['log_export', 'n_products', 'avg_rca', 'max_rca']:
    country_feat_df[col] = country_feat_df.groupby('year')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

country_feat_df.to_csv(country_output, index=False)

# Product Features per year
print("Creating product features per year...")
product_features = []
products = sorted(exports['product'].unique())

for year in years:
    year_data = exports[exports['year'] == year]
    # World export value for this product
    p_totals = year_data.groupby('product')['value'].sum()
    # Ubiquity: count countries with RCA >= 1
    p_ubiquity = year_data[year_data['rca'] >= 1].groupby('product').size()
    # Average RCA across countries for this product
    p_avg_rca = year_data.groupby('product')['rca'].mean()

    for p in products:
        features = {
            'year': year,
            'product': p,
            'log_world_export': np.log1p(p_totals.get(p, 0)),
            'ubiquity': p_ubiquity.get(p, 0),
            'avg_rca': p_avg_rca.get(p, 0),
        }
        product_features.append(features)

product_feat_df = pd.DataFrame(product_features)

# Normalize product features (z-score per year)
print("Normalizing product features...")
for col in ['log_world_export', 'ubiquity', 'avg_rca']:
    product_feat_df[col] = product_feat_df.groupby('year')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

product_feat_df.to_csv(product_output, index=False)

# Validation Statistics
print("\nValidation Statistics for Step 5:")
print(f"Country Features: {len(country_feat_df)} rows, saved to {country_output}")
print(f"Product Features: {len(product_feat_df)} rows, saved to {product_output}")
print(f"Unique features (country): {['log_export', 'n_products', 'avg_rca', 'max_rca']}")
print(f"Unique features (product): {['log_world_export', 'ubiquity', 'avg_rca']}")


Starting Step 5: Creating Node Features...
Creating country features per year...
Normalizing country features...
Creating product features per year...
Normalizing product features...

Validation Statistics for Step 5:
Country Features: 6990 rows, saved to data\country_features.csv
Product Features: 150540 rows, saved to data\product_features.csv
Unique features (country): ['log_export', 'n_products', 'avg_rca', 'max_rca']
Unique features (product): ['log_world_export', 'ubiquity', 'avg_rca']


In [4]:
import pandas as pd
import torch
import pickle
import os

# Define directories
output_dir = "data"
input_file = os.path.join(output_dir, "M_cpt_smoothed.csv")
edge_output = os.path.join(output_dir, "edge_index_by_year.pt")
country_map_output = os.path.join(output_dir, "country_mapping.pkl")
product_map_output = os.path.join(output_dir, "product_mapping.pkl")

print(f"Starting Step 6: Building Temporal Graph Snapshots...")

if not os.path.exists(input_file):
    print(f"Error: {input_file} not found. Did you run Step 3?")
    exit(1)

# Load edges (M=1)
edges = pd.read_csv(input_file)

# Create integer mappings for countries and products
# (Crucial: These mappings must be consistent across all steps and years)
print("Creating node mappings...")
country_list = sorted(edges['country'].unique())
product_list = sorted(edges['product'].unique())

country_to_idx = {c: i for i, c in enumerate(country_list)}
product_to_idx = {p: i for i, p in enumerate(product_list)}

idx_to_country = {i: c for c, i in country_to_idx.items()}
idx_to_product = {i: p for p, i in product_to_idx.items()}

# Save mappings
print(f"Saving mappings to {country_map_output} and {product_map_output}...")
with open(country_map_output, "wb") as f:
    pickle.dump({'to_idx': country_to_idx, 'to_name': idx_to_country}, f)
with open(product_map_output, "wb") as f:
    pickle.dump({'to_idx': product_to_idx, 'to_name': idx_to_product}, f)

# Build edge_index per year
# edge_index: [2, num_edges] for source (country) and target (product)
print("Building edge indexes per year...")
edge_index_by_year = {}
years = sorted(edges['year'].unique())

for year in years:
    year_edges = edges[edges['year'] == year]
    
    # Convert to integer indices
    country_idx = year_edges['country'].map(country_to_idx).values
    product_idx = year_edges['product'].map(product_to_idx).values
    
    # Edge index: [2, num_edges]
    edge_index = torch.tensor([country_idx, product_idx], dtype=torch.long)
    edge_index_by_year[year] = edge_index

# Save as .pt file
print(f"Saving edge indexes to {edge_output}...")
torch.save(edge_index_by_year, edge_output)

# Validation Statistics
print("\nValidation Statistics for Step 6:")
print(f"Output: {edge_output}")
print(f"Total years processed: {len(edge_index_by_year)}")
print(f"Num countries: {len(country_to_idx)}")
print(f"Num products: {len(product_to_idx)}")
sample_year = years[0]
print(f"Snapshot edges (year {sample_year}): {edge_index_by_year[sample_year].shape[1]}")


Starting Step 6: Building Temporal Graph Snapshots...
Creating node mappings...
Saving mappings to data\country_mapping.pkl and data\product_mapping.pkl...
Building edge indexes per year...


C:\Users\Ashwa\AppData\Local\Temp\ipykernel_40576\561026400.py:55: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index = torch.tensor([country_idx, product_idx], dtype=torch.long)


Saving edge indexes to data\edge_index_by_year.pt...

Validation Statistics for Step 6:
Output: data\edge_index_by_year.pt
Total years processed: 30
Num countries: 233
Num products: 5018
Snapshot edges (year 1995): 64896


In [5]:
import pandas as pd
import torch
import pickle
import os

# Define directories
output_dir = "data"
country_feat_file = os.path.join(output_dir, "country_features.csv")
product_feat_file = os.path.join(output_dir, "product_features.csv")
country_map_file = os.path.join(output_dir, "country_mapping.pkl")
product_map_file = os.path.join(output_dir, "product_mapping.pkl")

country_x_output = os.path.join(output_dir, "country_x_by_year.pt")
product_x_output = os.path.join(output_dir, "product_x_by_year.pt")

print(f"Starting Step 7: Creating Node Feature Tensors...")

if not all(os.path.exists(f) for f in [country_feat_file, product_feat_file, country_map_file, product_map_file]):
    print("Error: Required input files not found. Did you run previous steps?")
    exit(1)

# Load features and mappings
country_feat = pd.read_csv(country_feat_file)
product_feat = pd.read_csv(product_feat_file)

with open(country_map_file, "rb") as f:
    country_map = pickle.load(f)
with open(product_map_file, "rb") as f:
    product_map = pickle.load(f)

# Sort based on indices to ensure tensors match edge index indices
country_x_by_year = {}
product_x_by_year = {}

years = sorted(country_feat['year'].unique())

for year in years:
    # Country features for this year
    # We must ensure that row 'i' of our tensor is for country with index 'i'
    year_country = country_feat[country_feat['year'] == year].copy()
    year_country['idx'] = year_country['country'].map(country_map['to_idx'] or -1)
    # Filter out any that weren't in the mapping (though they should all be there)
    year_country = year_country.sort_values('idx')
    
    # Feature columns: log_export, n_products, avg_rca, max_rca
    country_x = torch.tensor(
        year_country[['log_export', 'n_products', 'avg_rca', 'max_rca']].values,
        dtype=torch.float
    )
    country_x_by_year[year] = country_x
    
    # Product features for this year
    year_product = product_feat[product_feat['year'] == year].copy()
    year_product['idx'] = year_product['product'].map(product_map['to_idx'] or -1)
    year_product = year_product.sort_values('idx')
    
    # Feature columns: log_world_export, ubiquity, avg_rca
    product_x = torch.tensor(
        year_product[['log_world_export', 'ubiquity', 'avg_rca']].values,
        dtype=torch.float
    )
    product_x_by_year[year] = product_x

# Save
print(f"Saving feature tensors to {country_x_output} and {product_x_output}...")
torch.save(country_x_by_year, country_x_output)
torch.save(product_x_by_year, product_x_output)

# Validation Statistics
print("\nValidation Statistics for Step 7:")
print(f"Num years: {len(years)}")
sample_year = years[0]
print(f"Country feature shape (year {sample_year}): {country_x_by_year[sample_year].shape}")
print(f"Product feature shape (year {sample_year}): {product_x_by_year[sample_year].shape}")


Starting Step 7: Creating Node Feature Tensors...
Saving feature tensors to data\country_x_by_year.pt and data\product_x_by_year.pt...

Validation Statistics for Step 7:
Num years: 30
Country feature shape (year 1995): torch.Size([233, 4])
Product feature shape (year 1995): torch.Size([5018, 3])


In [6]:
import pandas as pd
import os

# Define directories
output_dir = "data"
labels_file = os.path.join(output_dir, "labels_h5.csv")

train_output = os.path.join(output_dir, "train_labels.csv")
val_output = os.path.join(output_dir, "val_labels.csv")
test_output = os.path.join(output_dir, "test_labels.csv")

print(f"Starting Step 8: Splitting Into Train/Validation/Test Windows...")

if not os.path.exists(labels_file):
    print(f"Error: {labels_file} not found. Did you run Step 4?")
    exit(1)

# Load labels
labels = pd.read_csv(labels_file)

# Split according to plan:
# Training: up to 2012
# Validation: 2013
# Test: 2015
train_labels = labels[labels['year'].between(2000, 2012)]
val_labels = labels[labels['year'] == 2013]
test_labels = labels[labels['year'] == 2015]

# Save split labels
print(f"Saving splits to {train_output}, {val_output}, and {test_output}...")
train_labels.to_csv(train_output, index=False)
val_labels.to_csv(val_output, index=False)
test_labels.to_csv(test_output, index=False)

# Validation Statistics
print("\nValidation Statistics for Step 8:")
print(f"Train labels (2000-2012): {len(train_labels)} rows, balanced.")
print(f"Val labels (2013): {len(val_labels)} rows.")
print(f"Test labels (2015): {len(test_labels)} rows.")
print(f"Total entries: {len(train_labels) + len(val_labels) + len(test_labels)}")
print(f"Positive samples (Train): {(train_labels['label'] == 1).sum()}")
print(f"Negative samples (Train): {(train_labels['label'] == 0).sum()}")


Starting Step 8: Splitting Into Train/Validation/Test Windows...
Saving splits to data\train_labels.csv, data\val_labels.csv, and data\test_labels.csv...

Validation Statistics for Step 8:
Train labels (2000-2012): 837403 rows, balanced.
Val labels (2013): 61851 rows.
Test labels (2015): 62046 rows.
Total entries: 961300
Positive samples (Train): 278610
Negative samples (Train): 558793


In [10]:
%pip install torch_geometric


Defaulting to user installation because normal site-packages is not writeable
  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
Using cached torch_geometric-2.7.0-py3-none-any.whl (1.3 MB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)

   ------------- -------------------------- 3/9 [frozenlist]
   ------------------------------- -------- 7/9 [aiohttp]
   ------------------------------- -------- 7/9 [aiohttp]
   ------------------------------- -------- 7/9 [aiohttp]
   ------------------------------- -------- 7/9 [aiohttp]
   ----------------------------------- ---- 8/9 [torch_geometric]
   ----------------------------------- ---- 8/9 [torch_geometric]
   ----------------------------------- ---- 8/9 [torch_geometric]
   ----------------------------------- ----

In [14]:
import torch
import pandas as pd
import pickle
import os
from torch_geometric.data import HeteroData

# Define directories
output_dir = "data"
edge_file = os.path.join(output_dir, "edge_index_by_year.pt")
country_x_file = os.path.join(output_dir, "country_x_by_year.pt")
product_x_file = os.path.join(output_dir, "product_x_by_year.pt")
country_map_file = os.path.join(output_dir, "country_mapping.pkl")
product_map_file = os.path.join(output_dir, "product_mapping.pkl")

train_labels_file = os.path.join(output_dir, "train_labels.csv")
val_labels_file = os.path.join(output_dir, "val_labels.csv")
test_labels_file = os.path.join(output_dir, "test_labels.csv")

train_data_output = os.path.join(output_dir, "train_data.pt")
val_data_output = os.path.join(output_dir, "val_data.pt")
test_data_output = os.path.join(output_dir, "test_data.pt")

print(f"Starting Step 9: Creating PyTorch Geometric HeteroData Objects...")

def create_hetero_snapshot(year, country_x_by_year, product_x_by_year, edge_index_by_year):
    """Create a single HeteroData object for one year."""
    data = HeteroData()
    # Node features
    data['country'].x = country_x_by_year[year]
    data['product'].x = product_x_by_year[year]
    # Edges (country → product)
    data['country', 'exports', 'product'].edge_index = edge_index_by_year[year]
    return data

def create_temporal_batch(year, labels_df, country_x_by_year, product_x_by_year, edge_index_by_year, country_map, product_map):
    """
    Create a single predictive sample: 5-year history + future labels at year+5.
    (Self-note: labels at t+5 were generated from year 'year')
    """
    # History: 5 consecutive years ending at 'year'
    snapshots = [create_hetero_snapshot(y, country_x_by_year, product_x_by_year, edge_index_by_year) 
                 for y in range(year-4, year+1)]
    
    # Labels for year+5 (predicting what happens with labels marked for this 'year')
    sample_labels = labels_df[labels_df['year'] == year].copy()
    
    # Map country/product names to indices (must match feature/edge indexing)
    country_idx = sample_labels['country'].map(country_map['to_idx']).values
    product_idx = sample_labels['product'].map(product_map['to_idx']).values
    label_values = sample_labels['label'].values
    
    # edge_label_index: [2, num_labeled_edges]
    edge_label_index = torch.tensor([country_idx, product_idx], dtype=torch.long)
    edge_label = torch.tensor(label_values, dtype=torch.float)
    
    labels_dict = {
        'edge_label_index': edge_label_index,
        'edge_label': edge_label
    }
    
    return {'snapshots': snapshots, 'labels': labels_dict, 'year': int(year)}

# Load all data
print("Loading all node features and edges...")
edge_index_by_year = torch.load(edge_file, weights_only=False)
country_x_by_year = torch.load(country_x_file, weights_only=False)
product_x_by_year = torch.load(product_x_file, weights_only=False)

with open(country_map_file, "rb") as f:
    country_map = pickle.load(f)
with open(product_map_file, "rb") as f:
    product_map = pickle.load(f)

# Load labels split
train_labels = pd.read_csv(train_labels_file)
val_labels = pd.read_csv(val_labels_file)
test_labels = pd.read_csv(test_labels_file)

# Create training samples
print("Processing training samples...")
train_samples = []
for year in sorted(train_labels['year'].unique()):
    print(f"Creating sample for year {year} (requires history {year-4}-{year})...")
    sample = create_temporal_batch(year, train_labels, country_x_by_year, product_x_by_year, edge_index_by_year, country_map, product_map)
    train_samples.append(sample)

# Create validation sample
print("Processing validation sample (2013)...")
val_sample = create_temporal_batch(2013, val_labels, country_x_by_year, product_x_by_year, edge_index_by_year, country_map, product_map)

# Create test sample
print("Processing test sample (2015)...")
test_sample = create_temporal_batch(2015, test_labels, country_x_by_year, product_x_by_year, edge_index_by_year, country_map, product_map)

# Save
print(f"Saving final .pt data objects to {train_data_output}, {val_data_output}, and {test_data_output}...")
torch.save(train_samples, train_data_output)
torch.save(val_sample, val_data_output)
torch.save(test_sample, test_data_output)

# Validation Statistics
print("\nValidation Statistics for Step 9:")
print(f"Total training years: {len(train_samples)}")
print(f"Validation target year: {val_sample['year']}")
print(f"Test target year: {test_sample['year']}")
sample_year = train_samples[0]['year']
print(f"Sample (year {sample_year}): {len(train_samples[0]['snapshots'])} graph snapshots.")
print(f"Labeled edges for year {sample_year}: {train_samples[0]['labels']['edge_label'].shape[0]}")


Starting Step 9: Creating PyTorch Geometric HeteroData Objects...
Loading all node features and edges...
Processing training samples...
Creating sample for year 2000 (requires history 1996-2000)...
Creating sample for year 2001 (requires history 1997-2001)...
Creating sample for year 2002 (requires history 1998-2002)...
Creating sample for year 2003 (requires history 1999-2003)...
Creating sample for year 2004 (requires history 2000-2004)...
Creating sample for year 2005 (requires history 2001-2005)...
Creating sample for year 2006 (requires history 2002-2006)...
Creating sample for year 2007 (requires history 2003-2007)...
Creating sample for year 2008 (requires history 2004-2008)...
Creating sample for year 2009 (requires history 2005-2009)...
Creating sample for year 2010 (requires history 2006-2010)...
Creating sample for year 2011 (requires history 2007-2011)...
Creating sample for year 2012 (requires history 2008-2012)...
Processing validation sample (2013)...
Processing test sam

In [15]:
import torch
from torch_geometric.loader import DataLoader
from torch.utils.data import Dataset
import os

# Define directories
output_dir = "data"
train_data_file = os.path.join(output_dir, "train_data.pt")
val_data_file = os.path.join(output_dir, "val_data.pt")
test_data_file = os.path.join(output_dir, "test_data.pt")

class TemporalBipartiteDataset(Dataset):
    def __init__(self, samples):
        """
        samples: list of dicts with 'snapshots' and 'labels'
        """
        self.samples = samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

print(f"Starting Step 10: Initializing Data Loaders...")

if not os.path.exists(train_data_file):
    print(f"Error: {train_data_file} not found. Did you run Step 9?")
    exit(1)

# Load data
print(f"Loading data from {train_data_file}...")
train_samples = torch.load(train_data_file, weights_only=False)

# Create dataset
train_dataset = TemporalBipartiteDataset(train_samples)

# Create loader (batch_size=1, since each sample is a full year of graphs/labels)
print("Initializing DataLoader (batch_size=1)...")
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# Example: iterate through one batch
for batch in train_loader:
    snapshots = batch['snapshots']  # list of 5 graphs
    labels = batch['labels']  # dict with edge_label_index and edge_label
    year = batch['year']
    
    print(f"\nExample Batch Information:")
    print(f"Year: {year.item()}")
    print(f"Num snapshots in batch: {len(snapshots)}")
    print(f"Labeled edges (targets): {labels['edge_label'].shape[1] if len(labels['edge_label'].shape) > 1 else labels['edge_label'].shape[0]}")
    print(f"Example Edge Indices (first 5): {labels['edge_label_index'][:, :5]}")
    break

print("\nValidation Statistics for Step 10:")
print(f"DataLoader initialized for {len(train_dataset)} training years.")
print(f"Data objects are ready for model training.")


Starting Step 10: Initializing Data Loaders...
Loading data from data\train_data.pt...
Initializing DataLoader (batch_size=1)...

Example Batch Information:
Year: 2006
Num snapshots in batch: 5
Labeled edges (targets): 64901
Example Edge Indices (first 5): tensor([[[   0,    0,    0,  ...,  225,   17,   11],
         [  14,  255,  263,  ..., 1687, 1390,  633]]])

Validation Statistics for Step 10:
DataLoader initialized for 13 training years.
Data objects are ready for model training.
